[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · **[10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb)** · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 9 课：子图与多智能体 Handoff](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) ｜ [➡️ 下一课：第 11 课：生产特性（重试 / 缓存 / Runtime / 容错）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

---

# 第 10 课：Functional API —— @entrypoint + @task（不画图也能享受持久化/中断）

* @task：一个可被检查点记录的工作单元，调用返回 future，.result() 取值；可并行。
* @entrypoint(checkpointer=...)：工作流入口，用普通 Python 控制流（if/for/while）。
* 同样支持 interrupt()、Command(resume=...)、流式、previous（上一轮返回值）。
* 何时用：已有命令式代码想加上持久化/HITL；何时用 Graph API：需要可视化、复杂分支、并行状态合并。

In [1]:
#@title 🔧 第 0 步：初始化环境（在 Colab 中点击左侧 ▶ 即可自动安装依赖，本地运行自动跳过） { display-mode: "form" }
import importlib.util, subprocess, sys
if importlib.util.find_spec('langgraph') is None:
    print('⏳ 正在安装 LangGraph 环境（约 10 秒）...')
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'langgraph', 'langchain-core', 'langgraph-checkpoint-sqlite'])
    print('✅ 安装完成！')
else:
    import importlib.metadata as m
    print(f'✅ 环境已就绪：langgraph={m.version("langgraph")}')


✅ 环境已就绪：langgraph=1.2.22


In [2]:
import time

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.func import entrypoint, task
from langgraph.types import Command, interrupt


@task
def write_section(topic: str) -> str:
    time.sleep(0.3)  # 模拟耗时（并行时总耗时 ≈ 0.3s 而非 0.9s）
    return f"《{topic}》章节内容"


@task
def combine(sections: list[str]) -> str:
    return "\n".join(sections)


@entrypoint(checkpointer=InMemorySaver())
def report_workflow(topics: list[str]) -> dict:
    t0 = time.time()
    futures = [write_section(t) for t in topics]  # 并行启动
    sections = [f.result() for f in futures]
    draft = combine(sections).result()
    elapsed = time.time() - t0
    # 💡 恢复(resume)时整个 entrypoint 会从头重跑，但已完成的 @task 结果直接从检查点读取，
    #    不会再执行 —— 所以恢复后打印的 parallel_seconds ≈ 0。这正是 @task 的意义：
    #    把有副作用/耗时的操作包进 task，保证重放时不重复执行。

    feedback = interrupt({"draft": draft, "ask": "满意吗？输入修改意见或 'ok'"})
    if feedback != "ok":
        draft += f"\n（已根据意见修改：{feedback}）"
    return {"report": draft, "parallel_seconds": round(elapsed, 2)}


@entrypoint(checkpointer=InMemorySaver())
def counter(n: int, *, previous: int | None = None) -> int:
    # previous = 同一线程上一次的返回值 —— 函数式的"短期记忆"
    return (previous or 0) + n

## ▶️ 运行实验

In [3]:
cfg = {"configurable": {"thread_id": "f1"}}
for chunk in report_workflow.stream(["状态", "节点", "边"], cfg, stream_mode="updates"):
    print("  ", chunk)
print("\n恢复：")
print(report_workflow.invoke(Command(resume="多加一些例子"), cfg))

c = {"configurable": {"thread_id": "c"}}
print("\ncounter:", [counter.invoke(5, c) for _ in range(3)])

   {'write_section': '《状态》章节内容'}
   {'write_section': '《节点》章节内容'}
   {'write_section': '《边》章节内容'}
   {'combine': '《状态》章节内容\n《节点》章节内容\n《边》章节内容'}
   {'__interrupt__': (Interrupt(value={'draft': '《状态》章节内容\n《节点》章节内容\n《边》章节内容', 'ask': "满意吗？输入修改意见或 'ok'"}, id='e7e5011280a899c648ac286f5891954a', response_schema=None),)}

恢复：
{'report': '《状态》章节内容\n《节点》章节内容\n《边》章节内容\n（已根据意见修改：多加一些例子）', 'parallel_seconds': 0.0}

counter: [5, 10, 15]


## ✏️ 练习：把第 4 课的 ReAct agent 用 Functional API 重写（参考 docs/src/oss/langgraph/functional-api.mdx）。

In [4]:
# 在这里写你的练习代码 👇


[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb) [![GitHub](https://img.shields.io/badge/GitHub-仓库主页-181717?logo=github)](https://github.com/Luca5702/langgraph-study)

**📚 课程导航：** [01](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/01_hello_graph.ipynb) · [02](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/02_state_reducers.ipynb) · [03](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/03_control_flow.ipynb) · [04](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/04_agent_tools.ipynb) · [05](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/05_persistence_memory.ipynb) · [06](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/06_human_in_the_loop.ipynb) · [07](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/07_time_travel.ipynb) · [08](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/08_streaming.ipynb) · [09](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) · **[10](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/10_functional_api.ipynb)** · [11](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

[⬅️ 上一课：第 9 课：子图与多智能体 Handoff](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/09_subgraphs.ipynb) ｜ [➡️ 下一课：第 11 课：生产特性（重试 / 缓存 / Runtime / 容错）](https://colab.research.google.com/github/Luca5702/langgraph-study/blob/main/notebooks/11_production.ipynb)

---